# Experiment 11 – Image Generation Using GAN

### Deep Learning Laboratory – TensorFlow/Keras

**Aim:** To implement a Generative Adversarial Network (GAN) using TensorFlow/Keras for generating new handwritten digit images similar to the MNIST dataset.

### Learning Objectives
- Understand the basic concept of Generative AI.
- Understand the Generator and Discriminator networks.
- Learn how a GAN is trained using adversarial learning.
- Implement a simple GAN using TensorFlow/Keras.
- Generate and visualize synthetic images.


## 1. Theory

A **Generative Adversarial Network (GAN)** is a deep-learning model that learns to generate new data similar to the training data.

A GAN contains two neural networks:

### Generator
The Generator receives random noise and tries to create a realistic image.

### Discriminator
The Discriminator receives an image and predicts whether it is **real** (from the training dataset) or **fake** (created by the Generator).

### GAN Architecture

```text
Random Noise
     ↓
 Generator
     ↓
Fake Image ─────────┐
                    ↓
              Discriminator
                    ↑
Real Image ──────────┘
                    ↓
             Real / Fake
```

The Generator tries to fool the Discriminator, while the Discriminator tries to correctly identify real and generated images. This competition helps the Generator improve over time.

In [ ]:
# Step 1: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Load and Prepare the MNIST Dataset

The **MNIST dataset** contains grayscale images of handwritten digits from 0 to 9. Each image is 28 × 28 pixels.

For this experiment, the images are normalized to the range `[-1, 1]`, which works well with a Generator using `tanh` activation.

In [ ]:
# Step 2: Load MNIST dataset
(x_train, _), (_, _) = keras.datasets.mnist.load_data()

# Use a subset so the experiment runs faster in a student lab
x_train = x_train[:20000]

# Add channel dimension and normalize to [-1, 1]
x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

print("Training data shape:", x_train.shape)
print("Minimum value:", x_train.min())
print("Maximum value:", x_train.max())

In [ ]:
# Step 3: Display sample real images
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow((x_train[i].squeeze() + 1) / 2, cmap="gray")
    plt.axis("off")
plt.suptitle("Sample MNIST Images")
plt.show()

## 3. Create the Generator

The Generator starts with a random noise vector of length 100. Dense and convolutional layers gradually transform the noise into a 28 × 28 image.

`tanh` is used at the output because the training images were scaled to `[-1, 1]`.

In [ ]:
# Step 4: Build the Generator
NOISE_DIM = 100


def build_generator():
    model = keras.Sequential(name="Generator")
    model.add(layers.Input(shape=(NOISE_DIM,)))
    model.add(layers.Dense(7 * 7 * 128, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((7, 7, 128)))

    model.add(layers.Conv2DTranspose(64, kernel_size=5, strides=2,
                                     padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(1, kernel_size=5, strides=2,
                                     padding="same", activation="tanh"))
    return model


generator = build_generator()
generator.summary()

## 4. Create the Discriminator

The Discriminator is a binary classifier. It receives a 28 × 28 image and produces a probability indicating whether the image is real or generated.

In [ ]:
# Step 5: Build the Discriminator

def build_discriminator():
    model = keras.Sequential(name="Discriminator")
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Conv2D(64, kernel_size=5, strides=2, padding="same"))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, kernel_size=5, strides=2, padding="same"))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model


discriminator = build_discriminator()
discriminator.summary()

## 5. Define the Loss Function and Optimizers

The GAN uses **Binary Cross-Entropy** to measure how well the Discriminator distinguishes real and fake images.

Adam is used as the optimizer for both networks.

In [ ]:
# Step 6: Define loss and optimizers
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


generator_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

## 6. Create the GAN Training Step

One training step works as follows:

1. Generate random noise.
2. Generator creates fake images.
3. Discriminator evaluates real and fake images.
4. Calculate Generator and Discriminator losses.
5. Update both networks using backpropagation.

In [ ]:
# Step 7: One GAN training step
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(
        gen_loss, generator.trainable_variables
    )
    gradients_of_discriminator = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gradients_of_generator, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(gradients_of_discriminator, discriminator.trainable_variables)
    )

    return gen_loss, disc_loss

## 7. Prepare the Dataset

The MNIST images are shuffled and grouped into batches. A smaller batch size is used to make the notebook suitable for common Google Colab/student systems.

In [ ]:
# Step 8: Create TensorFlow dataset
BATCH_SIZE = 128

train_dataset = tf.data.Dataset.from_tensor_slices(x_train)
train_dataset = train_dataset.shuffle(len(x_train)).batch(BATCH_SIZE)

print("Number of batches:", tf.data.experimental.cardinality(train_dataset).numpy())

## 8. Train the GAN

GAN training can take time. For a laboratory demonstration, 10 epochs are used. The generated images should generally become more digit-like as training progresses.


In [ ]:
# Step 9: Train the GAN
EPOCHS = 10
num_examples_to_generate = 16
seed = tf.random.normal([num_examples_to_generate, NOISE_DIM])

generator_losses = []
discriminator_losses = []

for epoch in range(EPOCHS):
    gen_epoch_loss = []
    disc_epoch_loss = []

    for image_batch in train_dataset:
        gen_loss, disc_loss = train_step(image_batch)
        gen_epoch_loss.append(float(gen_loss))
        disc_epoch_loss.append(float(disc_loss))

    generator_losses.append(np.mean(gen_epoch_loss))
    discriminator_losses.append(np.mean(disc_epoch_loss))

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"Generator Loss: {generator_losses[-1]:.4f}, "
        f"Discriminator Loss: {discriminator_losses[-1]:.4f}"
    )

In [ ]:
# Step 10: Generate images after training
predictions = generator(seed, training=False)

plt.figure(figsize=(8, 8))
for i in range(num_examples_to_generate):
    plt.subplot(4, 4, i + 1)
    plt.imshow((predictions[i].numpy().squeeze() + 1) / 2, cmap="gray")
    plt.axis("off")
plt.suptitle("Images Generated by GAN")
plt.tight_layout()
plt.show()

In [ ]:
# Step 11: Plot Generator and Discriminator losses
plt.figure(figsize=(8, 5))
plt.plot(generator_losses, label="Generator Loss")
plt.plot(discriminator_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Loss")
plt.legend()
plt.show()

## 9. Generate New Images from Random Noise

After training, the Generator can create new images from previously unseen random noise vectors. These images are not copied directly from the training dataset; the Generator learns patterns that help it create digit-like images.

In [ ]:
# Step 12: Generate a fresh set of images
new_noise = tf.random.normal([16, NOISE_DIM])
new_images = generator(new_noise, training=False)

plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow((new_images[i].numpy().squeeze() + 1) / 2, cmap="gray")
    plt.axis("off")
plt.suptitle("New Synthetic MNIST-like Images")
plt.tight_layout()
plt.show()

## 10. Student Practice

Try the following experiments:

1. Change `EPOCHS` from 10 to 20 and observe the generated images.
2. Change `NOISE_DIM` and modify the Generator accordingly.
3. Change the Generator learning rate from `0.0002` to `0.0001`.
4. Compare generated images before and after additional training.
5. Explain why both Generator and Discriminator are needed.

### Observation Table

| Configuration | Observation |
|---|---|
| 10 epochs | __________________ |
| 20 epochs | __________________ |
| Learning rate = 0.0001 | __________________ |

## Result

Thus, a **Generative Adversarial Network (GAN)** was successfully implemented using TensorFlow/Keras. A Generator and Discriminator were trained adversarially, and the trained Generator produced synthetic handwritten digit-like images.

## Viva Questions

1. What is a GAN?
2. What is the role of the Generator?
3. What is the role of the Discriminator?
4. Why is random noise given to the Generator?
5. What is adversarial training?
6. Why is MNIST used in this experiment?
7. What is the purpose of the `tanh` activation in the Generator?
8. What is Binary Cross-Entropy?
9. What happens if the Discriminator becomes too strong?
10. Give two real-world applications of GANs.